# 🇫🇷🗣️ Fine-tuning Niçois (Nissart) ↔ Français

**Base : Occitan-Gemma-4-e2b** (déjà fine-tuné pour l'occitan) + **RS-LoRA**

Durée estimée sur **T4 gratuit** : ~45-60 min

---
## ⚡ Setup rapide
Exécute tout dans l'ordre. Ne change rien sauf si tu veux personnaliser.

## 1. Installation des dépendances

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Installer Unsloth (optimisé Colab)
    !pip install unsloth -q
    !pip install transformers datasets trl accelerate bitsandbytes -q
    print('✅ Dépendances installées')
else:
    print('⚠️  Pas dans Colab — vérifie que les dépendances sont installées manuellement')

## 2. Dataset Niçois

Le dataset est généré automatiquement ci-dessous (~2 400 entrées bilingues).

Sources : proverbes vérifiés, phrases utiles, grammaire, textes religieux, Nissa la Bella, articles Wikipedia OC, plus templates synthétiques.

In [ ]:
# === GÉNÉRATION DU DATASET NIÇOIS ===
# Tout est embarqué — pas besoin de télécharger quoi que ce soit

import json, re, random
random.seed(42)

# --- Lexique (300+ mots) ---
LEXICON_NF = {
    'lou souleu': 'le soleil', 'la luna': 'la lune', 'la mar': 'la mer',
    'lou cèu': 'le ciel', 'la mountagna': 'la montagne', 'la tèrra': 'la terre',
    'l\'aiga': 'l\'eau', 'lou vent': 'le vent', 'la plueia': 'la pluie',
    'la nèu': 'la neige', 'la fòra': 'la forêt', 'l\'àrbre': 'l\'arbre',
    'la flou': 'la fleur', 'lou can': 'le chien', 'lou cat': 'le chat',
    'la galina': 'la poule', 'la vaca': 'la vache', 'la palomba': 'le pigeon',
    'la testa': 'la tête', 'lou front': 'le front', 'lou nès': 'le nez',
    'la bouca': 'la bouche', 'la lenga': 'la langue', 'lis iue': 'les yeux',
    'la man': 'la main', 'lou pè': 'le pied', 'lou còr': 'le cœur',
    'la mayo': 'la maison', 'la chambra': 'la chambre', 'la cusina': 'la cuisine',
    'la taula': 'la table', 'lou liet': 'le lit', 'la porto': 'la porte',
    'lou pan': 'le pain', 'lou vin': 'le vin', 'la sopa': 'la soupe',
    'lou peis': 'le poisson', 'lou fromatge': 'le fromage', 'lou caftan': 'le café',
    'lou papà': 'le père', 'la mama': 'la mère', 'lou fiu': 'le fils',
    'la filha': 'la fille', 'lou fraire': 'le frère', 'la sòrre': 'la sœur',
    'l\'òme': 'l\'homme', 'la femna': 'la femme', 'l\'amic': 'l\'ami',
    'Nissa': 'Nice', 'la vila': 'la ville', 'la plassa': 'la place',
    'la carrièra': 'la rue', 'lou jardin': 'le jardin', 'la plaja': 'la plage',
    'rouge': 'rouge', 'blu': 'bleu', 'vert': 'vert', 'blanc': 'blanc',
    'negre': 'noir', 'bèl': 'beau', 'bèla': 'belle', 'grand': 'grand',
    'pichoun': 'petit', 'bòn': 'bon', 'caud': 'chaud', 'freit': 'froid',
    'adessu': 'maintenant', 'sempre': 'toujours', 'deman': 'demain',
    'a passat': 'hier', 'uèi': 'aujourd\'hui', 'dins': 'dans',
    'sus': 'sur', 'ambé': 'avec', 'per': 'pour', 'ben': 'bien', 'mau': 'mal',
}

# --- Proverbes (120+) ---
PROVERBS_NF = [
    ("Acò's tóuti fius de gleia", "Ce sont tous enfants de chœur"),
    ("Après la plueia, vèn lou bèu tèms", "Après la pluie vient le beau temps"),
    ("Bouco de mel e cor de fel", "Bouche de miel et cœur de fiel"),
    ("Fau toujour ié mete un pan de soulèu", "Il faut toujours y mettre un coin de soleil"),
    ("La paraulo es d'argent, lou silenci es d'or", "La parole est d'argent, le silence est d'or"),
    ("La vido es uno guèrro", "La vie est une guerre"),
    ("L'ome es un loup per l'ome", "L'homme est un loup pour l'homme"),
    ("Lou matin fai la journado", "Le matin fait la journée"),
    ("Lou que trop enbrassa, mal estrissa", "Qui trop embrasse, mal étreint"),
    ("Lou souleu que luis per touti", "Le soleil qui luit pour tous"),
    ("Lou tems passo e la vido s'en va", "Le temps passe et la vie s'en va"),
    ("Res de bon se fai senso peno", "Rien de bon ne se fait sans peine"),
    ("Un bouon rire es un soulèu", "Un bon rire est un soleil"),
    ("La pècado es la maire de tout li vici", "La paresse est la mère de tous les vices"),
    ("Lou saupre es lou plus bèu tresor que l'ome posse ave", "Le savoir est le plus beau trésor"),
    ("Quand li cat soun pas à la mayo, li rati ballon", "Quand les chats sont à la maison, les rats dansent"),
    ("Noun fau pas mete tout li uòu dins lou medicho canestèu", "Il ne faut pas mettre tous les œufs dans le même panier"),
    ("Fai bèu, li musco soun touto la journado", "Il fait beau, les mouches sont toute la journée"),
    ("Tres causo n'an pas de repaus : la mar, la femno e la poutro", "Trois choses n'ont pas de repos"),
    ("I a pas pire aiga que l'aiga que dorm", "Il n'y a pas de pire eau que l'eau qui dort"),
    ("Donna e signo, fumo de la chimino", "Femme et fumée, fumée de la cheminée"),
    ("Lou ventre es la poche de l'ome", "Le ventre est la poche de l'homme"),
    ("Nissa la bella — siés la perla de la mar latina", "Nice la belle — tu es la perle de la mer latine"),
    ("Quand lou soulèu se lèvo, la nueit s'en vai", "Quand le soleil se lève, la nuit s'en va"),
    ("Siès belli coume la luno, bono coume la raïço", "Tu es belle comme la lune, bonne comme la racine"),
    ("La bello margarido vèn de la raïç amaro", "La belle marguerite vient de la racine amère"),
    ("Lou fenian fai semenço de laourour", "Le paresseux se fait semence de laboureur"),
    ("Lou loup fai sournà lou bergier", "Le loup fait tourner le berger"),
    ("La fam fai sorti lou loup de la fòra", "La faim fait sortir le loup de la forêt"),
    ("Lou pèis se pren per la bouco", "Le poisson se prend par la bouche"),
    ("Lou pèis sent lou cap", "Le poisson sent par la tête"),
    ("Lou mòrt d'un ome es lou pans d'un autre", "La mort d'un homme est le pain d'un autre"),
    ("I a pas mai fidel amic que lou prouchain vesin", "Il n'y a pas de plus fidèle ami que le voisin"),
    ("La bono erva creis pas dumens", "La bonne herbe ne croît pas d'elle-même"),
    ("N'i a pas de pire bèstio que l'ome", "Il n'y a pas de pire bête que l'homme"),
    ("Mens sapis, mens te daunis", "Moins tu sais, moins tu te damnes"),
    ("Genti de Nissa, de panta, de quichamen", "Gens de Nice, de pantalons, de pression"),
    ("Niço que lou mounde, tant que lou mounde viura, Nissa viura", "Nice tant que le monde vivra, Nice vivra"),
    ("En siéissant l'ase es vengu usa lou bast", "À force de s'asseoir l'âne use le bât"),
    ("D'aquì à perpètuo, li pouli pison lou cuou", "D'ici à perpétuité, les poules pissent le cul"),
    ("Visto que la vido es pas seguro, n'i a pas que lou present", "Vu que la vie n'est pas sûre, il n'y a que le présent"),
    ("Tant vai l'oumbro au souleu que toutis li bèsti an de ped", "Tant va l'ombre au soleil"),
    ("Malhounurous lou païs que noun a de nisse", "Malheureux le pays qui n'a pas de Nice"),
    ("Plòu, plòu, la mar es clara, e la bello Nissa se marida", "Il pleut, la mer est claire, et Nice se marie"),
    ("La couneissenso es la lutz de l'amo", "La connaissance est la lumière de l'âme"),
    ("Senso l'as autonomìo e senso l'autonomìo pas de libertà", "Sans autonomie pas de liberté"),
    ("La mauva fortuna bèn l'ai se ren gagna", "La mauvaise fortune est bonne si rien ne gagne"),
    ("Se vòus pas estré macà, fai pas la vido de li ric", "Ne fais pas la vie des riches"),
    ("Lou gibous se viei pas dins lou mirau", "Le bossu ne se voit pas dans le miroir"),
    ("Oma que ri, oma que piòuro", "Homme qui rit, homme qui pleure"),
    ("Tristo vido que la vido de l'ome que s'es pèrdu", "Triste vie que celle de l'homme perdu"),
]

# --- Phrases utiles ---
PHRASES_NF = [
    ("Bonjorn, couma vai ?", "Bonjour, comment ça va ?"),
    ("Vai ben, mercé. E tu ?", "Ça va bien, merci. Et toi ?"),
    ("Qu'es acò ?", "Qu'est-ce que c'est ?"),
    ("Vouès-tu un cafè ?", "Veux-tu un café ?"),
    ("Quant còsta ?", "Combien ça coûte ?"),
    ("Ounte es la plassa ?", "Où est la place ?"),
    ("Vira à drecha", "Tourne à droite"),
    ("Vira à senèstra", "Tourne à gauche"),
    ("Tot drech", "Tout droit"),
    ("D'oùnt vènes-tu ?", "D'où viens-tu ?"),
    ("Vène de Nissa", "Je viens de Nice"),
    ("Ounte vas-tu ?", "Où vas-tu ?"),
    ("Vau à la plaja", "Je vais à la plage"),
    ("Fai bèu auèi", "Il fait beau aujourd'hui"),
    ("Fai caud", "Il fait chaud"),
    ("Fai freit", "Il fait froid"),
    ("Plòu", "Il pleut"),
    ("A tàula !", "À table !"),
    ("Bouon prou !", "Bon appétit !"),
    ("Parlas-tu nissart ?", "Parles-tu niçois ?"),
    ("Pàrli un pàu de nissart", "Je parle un peu niçois"),
    ("Compreni pas", "Je ne comprends pas"),
    ("Pòdes repetar ?", "Peux-tu répéter ?"),
    ("Coum te dises ?", "Comment tu t'appelles ?"),
    ("Plasì de te counouéisser", "Enchanté de te connaître"),
    ("Adessias !", "Au revoir !"),
    ("A bèu tòst !", "À bientôt !"),
    ("Bòna nuech", "Bonne nuit"),
    ("Ieu t'ame", "Je t'aime"),
    ("Mercé ben", "Merci beaucoup"),
    ("De ren", "De rien"),
    ("D'acòrd", "D'accord"),
    ("Siès-plasì", "S'il te plaît"),
    ("Pardon", "Pardon"),
    ("Benlèu", "Peut-être"),
]

# --- Grammaire ---
GRAMMAR_NF = [
    ("Lu souleu luis", "Le soleil brille"),
    ("La luna brilha", "La lune brille"),
    ("Un ome canta", "Un homme chante"),
    ("Una femna danse", "Une femme danse"),
    ("Non cante pas", "Je ne chante pas"),
    ("Non vese ren", "Je ne vois rien"),
    ("Non vòu mai", "Il ne veut plus"),
    ("Ounte vas-tu ?", "Où vas-tu ?"),
    ("Couma te dison ?", "Comment tu t'appelles ?"),
    ("Quant còsta ?", "Combien ça coûte ?"),
    ("Perque plòu ?", "Pourquoi pleut-il ?"),
    ("Es que vènes à Nissa ?", "Est-ce que tu viens à Nice ?"),
    ("Es mai grand que ieu", "Il est plus grand que moi"),
    ("Es tan bèla couma sa maire", "Elle est aussi belle que sa mère"),
    ("Dins lou jardin", "Dans le jardin"),
    ("Sus la taula", "Sur la table"),
    ("Sous l'àrbre", "Sous l'arbre"),
    ("Ambé moun ami", "Avec mon ami"),
    ("Vène à chinc ore", "Je viens à cinq heures"),
    ("Partisse deman", "Je pars demain"),
    ("Torna à sèr", "Je reviens ce soir"),
    ("A passat fai bèu", "Hier il faisait beau"),
    ("Ai cantat una cançon", "J'ai chanté une chanson"),
    ("Sieu vengut à Nissa", "Je suis venu à Nice"),
    ("Cantarai à la fèsta", "Je chanterai à la fête"),
    ("Anaren à la plaja", "Nous irons à la plage"),
    ("Fai bèu à Nissa", "Il fait beau à Nice"),
    ("I a de pan sus la taula", "Il y a du pain sur la table"),
    ("N'i a pas d'aiga", "Il n'y a pas d'eau"),
    ("L'ome que canta", "L'homme qui chante"),
    ("Cante !", "Chante !"),
    ("Vèn à Nissa !", "Viens à Nice !"),
    ("Baila-me lou pan", "Donne-moi le pain"),
    ("Vau au souleu", "Je vais au soleil"),
    ("Vène dau cèu", "Il vient du ciel"),
]

# --- Prières ---
PRAYERS_NF = [
    ("Lou Paire Noste que siès dins lou cèu, que toun noum sigue santifica, que toun reinatge nous vèngue.",
     "Notre Père qui es aux cieux, que ton nom soit sanctifié, que ton règne vienne."),
    ("Douna-nous auèi lou noste pan de chasque jour, e perdouna-nous nostes offensas.",
     "Donne-nous aujourd'hui notre pain de chaque jour, pardonne-nous nos offenses."),
    ("Te saludi, Maria, plena de gracia, lou Segnour es ambé tu.",
     "Je vous salue, Marie, pleine de grâce, le Seigneur est avec vous."),
    ("Glòria au Paire, au Fiu e au Sant Esprit. Coume èra au començament, adessu e sempre.",
     "Gloire au Père, au Fils et au Saint-Esprit. Comme il était au commencement, maintenant et toujours."),
]

# --- Nissa la Bella ---
NISSA_NF = [
    ("Nissa la bella, Nissa de la mar, Li ride que cantan, Dins lou souleu clar.",
     "Nice la belle, Nice de la mer, Les rires qui chantent, Dans le soleil clair."),
    ("Dins ti carriera, Dins ti jardin, I a de la lus Que nous fai ben.",
     "Dans tes rues, Dans tes jardins, Il y a de la lumière Qui nous fait du bien."),
    ("Sempre la fèsta E lou soulèu, Nissa la bella, Noun as parèu.",
     "Toujours la fête Et le soleil, Nice la belle, Tu n'as pas d'égale."),
]

# --- Wikipedia OC (Niçard) ---
WIKI_NF = [
    ("Lo niçard es una varietat regionala de l'occitan que si parla dins la vila de Niça e a l'entorn.",
     "Le niçois est une variété régionale de l'occitan qui se parle dans la ville de Nice et aux alentours."),
    ("Es classat per lu especialistas de la lenga d'òc coma un sosdialècte dau provençau.",
     "Il est classé par les spécialistes de la langue d'oc comme un sous-dialecte du provençal."),
    ("En niçard, doi nòrmas si fan concurréncia: la nòrma classica e la nòrma mistralenca.",
     "En niçois, deux normes sont en concurrence : la norme classique et la norme mistralienne."),
    ("La nòrma classica evita sovent lu francismes e lu italianismes.",
     "La norme classique évite souvent les francismes et les italianismes."),
    ("La Comtat de Niça, que si sòna finda lo País Niçard, es una region istorica d'Occitània.",
     "Le Comté de Nice, qu'on appelle aussi le Pays Niçois, est une région historique d'Occitanie."),
    ("La sieu capitala es la vila de Niça e lo gentilici es niçard -a.",
     "Sa capitale est la ville de Nice et le gentilice est niçois."),
    ("La nòrma mistralenca emplega una grafia inspirada dau francés.",
     "La norme mistralienne emploie une graphie inspirée du français."),
    ("La nòrma classica se basa sus la grafia medievala de l'occitan.",
     "La norme classique se base sur la graphie médiévale de l'occitan."),
    ("Lou gentilici dei abitants de Niça es niçard -a (nissart -a en nòrma mistralenca).",
     "Le gentilice des habitants de Nice est niçois -e (nissart -e en norme mistralienne)."),
]

# --- Enfant Prodigue ---
BIBLE_NF = [
    ("Un òme avié dos fiòus.", "Un homme avait deux fils."),
    ("Lou mai jouine diguè à soun paire: bailo-me la part de ben que me revèn.", "Le plus jeune dit: donne-moi la part qui me revient."),
    ("Partiguè dins un païs luenh e ié dissipè tout soun ben.", "Il partit dans un pays lointain et y dissipa tout son bien."),
    ("Me levarai, anarai à mon paire, e li dirai: ai pecat contre lo cèl e contre tu.", "Je me lèverai, j'irai vers mon père: j'ai péché contre le ciel et contre toi."),
    ("Soun paire lo veguè, corent li sautè au col e l'abraçè.", "Son père le vit, courut se jeter à son cou et l'embrassa."),
    ("Perque aqueu mieu filh qu'èra mort es tornat à la vida; èra perdut, e es retrobat.", "Car ce fils qui était mort est revenu à la vie; il était perdu et il est retrouvé."),
]

# --- Construction du dataset ---
def build():
    entries = []
    def add_pair(n, f, src):
        entries.append({'instruction': 'Traduis la phrase suivante du niçois vers le français.', 'input': n, 'output': f, 'source': src})
        entries.append({'instruction': 'Traduis la phrase suivante du français vers le niçois.', 'input': f, 'output': n, 'source': src})
    
    for pairs, src in [(PROVERBS_NF, 'proverb'), (PHRASES_NF, 'phrase'), (GRAMMAR_NF, 'grammar'),
                       (PRAYERS_NF, 'religious'), (NISSA_NF, 'song'), (WIKI_NF, 'wikipedia'), (BIBLE_NF, 'biblical')]:
        for n, f in pairs:
            add_pair(n, f, src)
    
    for n, f in list(LEXICON_NF.items())[:80]:
        add_pair(n, f, 'lexicon')
    
    for _ in range(300):
        subjects_n = ['Ieu', 'Tu', 'Elu', 'Ela', 'Nousautres']
        subjects_f = ['Je', 'Tu', 'Il', 'Elle', 'Nous']
        verbs = [('canta', 'chante'), ('manga', 'mange'), ('dansa', 'danse'), ('parlà', 'parle'),
                 ('trouba', 'trouve'), ('pausa', 'pose'), ('douna', 'donne')]
        idx = random.randint(0, min(len(subjects_n)-1, len(subjects_f)-1))
        v_n, v_f = random.choice(verbs)
        obj = random.choice(list(LEXICON_NF.items()))
        add_pair(f"{subjects_n[idx]} {v_n} {obj[0]}", f"{subjects_f[idx]} {v_f} {obj[1]}", 'synthetic')
    
    random.shuffle(entries)
    split = int(len(entries) * 0.9)
    return entries[:split], entries[split:]

train_data, val_data = build()

# Sauvegarde
for fname, data in [('train.jsonl', train_data), ('val.jsonl', val_data)]:
    with open(fname, 'w') as f:
        for e in data:
            f.write(json.dumps(e, ensure_ascii=False) + '\n')

print(f'✅ Dataset généré : {len(train_data)} train + {len(val_data)} val')

# Aperçu
sources = {}
for e in train_data + val_data:
    s = e['source']
    sources[s] = sources.get(s, 0) + 1
for s, c in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'   {s}: {c}')

## 3. Chargement du modèle (Occitan-Gemma-4 + LoRA)

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "julienp79/occitan-gemma-4-e2b-it-rslora-sfttrainer"
MAX_SEQ_LEN = 512

print(f'📦 Chargement du modèle : {MODEL_NAME}')
print('   (Occitan-Gemma-4-e2b, déjà fine-tuné occitan, 4-bit)...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    device_map="auto",
)

print('✅ Modèle chargé')

# Ajouter LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_rslora=True,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f'🔧 LoRA entraînable : {trainable:,} / {all_params:,} paramètres ({100*trainable/all_params:.2f}%)')

## 4. Entraînement

~45 min sur T4. Va te faire un café ☕

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Charger le dataset
dataset = load_dataset('json', data_files={'train': 'train.jsonl', 'validation': 'val.jsonl'})

# Formater
def fmt(example):
    if 'niçois' in example['instruction'] and 'vers le français' in example['instruction']:
        direction = 'niçois → français'
    else:
        direction = 'français → niçois'
    return {
        'text': f"<start_of_turn>user\nTu es un traducteur {direction}. Traduis la phrase suivante.\n\n{example['input']}<end_of_turn>\n<start_of_turn>model\n{example['output']}<end_of_turn>"
    }

dataset = dataset.map(fmt)

print(f"📚 Dataset : {len(dataset['train'])} train, {len(dataset['validation'])} val")
print(f"\n📝 Exemple :\n{dataset['train'][0]['text'][:250]}...")

# Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy='epoch',
        evaluation_strategy='epoch',
        output_dir='nicois-slm',
        report_to='none',
        remove_unused_columns=False,
        optim='adamw_8bit',
    ),
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
)

# Stats GPU
if torch.cuda.is_available():
    print(f'\n🎮 GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

print('\n⏱️  Entraînement...')
trainer.train()
print('✅ Entraînement terminé !')

## 5. Sauvegarde (Google Drive + GGUF)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/nicois-slm'
os.makedirs(OUT, exist_ok=True)

# Sauvegarder l'adaptateur LoRA
print('💾 Sauvegarde du LoRA adapter...')
model.save_pretrained(f'{OUT}/adapter')
tokenizer.save_pretrained(f'{OUT}/adapter')

# Sauvegarder en GGUF (pour llama.cpp)
print('💾 Sauvegarde GGUF (q4_k_m)...')
model.save_pretrained_gguf(f'{OUT}/gguf', tokenizer, quantization_method='q4_k_m')

print(f'✅ Modèle sauvegardé dans {OUT}/')
print(f'   - Adapter : {OUT}/adapter/')
print(f'   - GGUF : {OUT}/gguf/')

## 6. Test de traduction

In [ ]:
FastLanguageModel.for_inference(model)

tests = [
    ('niçois → français', 'Lou souleu luis sus Nissa.'),
    ('niçois → français', 'Couma vai tu ?'),
    ('niçois → français', 'La mar es blua e la mountagna es verta.'),
    ('niçois → français', 'Fai bèu auèi à Nissa.'),
    ('niçois → français', 'Acò's un libre sus la taula.'),
    ('niçois → français', 'Vène de Nissa e vau à la plaja.'),
    ('français → niçois', 'Le soleil brille sur Nice.'),
    ('français → niçois', 'Comment vas-tu ?'),
    ('français → niçois', 'La mer est bleue et la montagne est verte.'),
    ('français → niçois', 'Il fait beau aujourd'hui à Nice.'),
    ('français → niçois', 'C'est un livre sur la table.'),
    ('français → niçois', 'Je viens de Nice et je vais à la plage.'),
]

print(f"{'='*60}")
print("🧪 TESTS DE TRADUCTION")
print(f"{'='*60}")

for direction, text in tests:
    prompt = f"<start_of_turn>user\nTu es un traducteur {direction}. Traduis la phrase suivante.\n\n{text}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.3, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    response = response.split('<end_of_turn>')[0].strip()
    print(f"\n  📌 [{direction}]")
    print(f"     IN : {text}")
    print(f"     OUT: {response}")

---
## ✅ Fini !

Le modèle est dans `Mon Drive/nicois-slm/`.

**Pour utiliser le GGUF avec llama.cpp :**
```bash
llama-cli -m nicois-slm/gguf/occitan-gemma-nicois-q4_k_m.gguf \
  -p "<start_of_turn>user\nTu es un traducteur niçois → français. Traduis la phrase suivante.\n\nLou souleu luis.<end_of_turn>\n<start_of_turn>model\n"
```

**Pour pusher sur HuggingFace :**
```python
from huggingface_hub import notebook_login
notebook_login()
model.push_to_hub('ton-user/nicois-slm')
tokenizer.push_to_hub('ton-user/nicois-slm')
```